# Extract mantle data

This notebook will extract mantle diagnostics from plate-model-driven G-ADOPT outputs, using the training points extracted by notebooks `00b` or `00bb`. The resultant data is saved or appended to `training_data_global_mantle.csv`. This can then be used to train the models in later notebooks (`01*.ipynb`).

G-ADOPT outputs have been interpolated into (lat, lon, depth, time) grids from original volumetric .xy files.

## Notebook options

These cells set some of the important variables and definitions used throughout the notebook.

In [1]:
config_file = "config/.run_config.yml"

In [2]:
from lib.load_params import get_params
from pathlib import Path

params = get_params(config_file, notebook="00d")

# =====================
# Plate model
# =====================

plate_model_name = params["plate_model"]["plate_model_name"]
use_provided_plate_model = params["plate_model"]["use_provided_plate_model"]

# =====================
# Filestructure
# =====================

# Parent data directory
source_data_dir = Path(params["data_dir"])

# Directory for data derived from chosen reconstruction
extracted_data_dir = source_data_dir / plate_model_name

# Directory for extracted point data
output_dirname = f"{params["reference_feature"]}_{params["study_zone_buffer"]:.1f}_deg_buffer"
output_dir = extracted_data_dir / "extracted_data" / output_dirname

# Plate model directory
plate_model_dir = extracted_data_dir / "plate_model"

# CSV file with known deposits; columns:
# lon, lat, age (Ma), label, source
deposits_filename = source_data_dir / "deposits" / params["deposits_filename"]

# If desired, categorise deposits according to location
# Should be a shapefile or GeoJSON containing polygons
# with a 'region' attribute
regions_filename = source_data_dir / params["regions_filename"]

# Initialise filestructure
source_data_dir.mkdir(parents=True, exist_ok=True)
extracted_data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

# =====================
# Run parameters
# =====================

# If True, append mantle information to training data extracted in notebook 00b, 00c
# Else, use pre-prepared training data downloaded from Zenodo
use_extracted_data = params["use_extracted_data"]

# Number of processes to use
n_jobs = params["n_jobs"]

# Overwrite any existing output files
overwrite = params["overwrite_output"]

# Control verbosity level of logging output
verbose = params["verbose"]

# Timespan for analysis
min_time = params["timespan"]["min"]
max_time = params["timespan"]["max"]
times = range(min_time, max_time + 1)

# Random seed for reproducibility
random_seed = params["random_seed"]


## Notebook setup

Imports, definitions, etc.

### Imports

In [3]:
import os
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.check_files import (
    check_prepared_data,
    check_plate_model,
)
from lib.misc import format_feature_name

from lib.plate_models import get_plate_reconstruction

from lib.sample_mantle import extract_basic_mantle_features

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning
warnings.simplefilter("ignore", UserWarning)

env: PYTHONWARNINGS=ignore::UserWarning


### Input and output files

If necessary, the plate model will be downloaded:

In [4]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

2026-03-23,22:30:07 - pmm - WARNING - Unable to fetch https://repo.gplates.org/webdav/pmm/config/models_v2.json.
2026-03-23,22:30:07 - pmm - WARNING - Unable to fetch https://www.earthbyte.org/webdav/pmm/config/models_v2_eb.json.
2026-03-23,22:30:07 - pmm - WARNING - Unable to fetch https://portal.gplates.org/static/pmm/config/models_v2_gp.json.


In [5]:
if use_extracted_data:
    data_dir = output_dir
    if verbose:
        print(f"Using extracted training data in {data_dir}")
else:
    data_dir = check_prepared_data("prepared_data", verbose=True)

# Directory for interpolated G-ADOPT output grids
mantle_data_dir = source_data_dir / "g-adopt-outputs" / plate_model_name / "nc_output"

# Use old data without mantle features if it exists, else use unmodified global training data
_candidate = data_dir / "training_data_no_mantle.csv"
data_filename = _candidate if _candidate.exists() else source_data_dir / "training_data_global.csv"

output_filename = output_dir / "training_data_with_mantle.csv"


Using extracted training data in /scratch/xd2/me5758/PUB-framework-Alfonso/data/zahirovic2022/extracted_data/trenches_6.0_deg_buffer


In [6]:
# Load training data
training_data = pd.read_csv(data_filename)

In [7]:
labels = training_data.label.unique()
for label in labels:
    n = training_data.loc[training_data["label"] == label, "plate_id"]
    print(f"{label}: {n.size} samples")
    print(f"{label}: {n.count()} with plate_id")

positive: 1355 samples
positive: 1355 with plate_id
negative: 1703 samples
negative: 1703 with plate_id
unlabelled: 42118 samples
unlabelled: 0 with plate_id


### Extract simple mantle features

Reconstruct labelled points to sample mantle model outputs at various depths. Resultant data are appended to `training_data_global.csv`. When sampling, linear interpolation is used to minimise distortion from temporal sparseness of the mantle grids.

In [ ]:
mantle_fields = (
    # "FullTemperature_CG",
    "Pressure",
    "Radial_Velocity",
    # "Temperature_CG",
    "Temperature_Deviation_CG",
    "Velocity_x",
    "Velocity_y",
    "Velocity_z",
    "Viscosity_CG",
)

training_data = extract_basic_mantle_features(
    mantle_dir=mantle_data_dir,
    points=training_data,
    mantle_fields=mantle_fields,
)

### Extract cumulative mantle features

This cell will reconstruct labelled points to calculate cumulative mantle diagnostics at various depths of the mantle model outputs. Resultant data are appended to `training_data_global.csv`. Linear interpolation is used to sample these outputs to minimise distortion from temporal sparseness of the mantle grids.

In [9]:
def extract_cumulative_mantle_features():
    ...

### Extract other shenanigans

And more, and more, and more!! What joy we have, living like leeches mawed to the capillaries of progress.

In [10]:
def extract_some_other_nonsense_too():
    ...

### Save to file

Finally, we write the dataset to a CSV file.

In [11]:
training_data.to_csv(output_filename, index=False)

training_data.groupby(["region", "label"]).size()

region          label     
East Asia       negative        14
                positive       159
                unlabelled    7729
North America   negative        61
                positive       293
                unlabelled    8104
Other           negative       214
                positive         2
                unlabelled    6476
South America   negative      1389
                positive       275
                unlabelled    7845
Southeast Asia  negative         4
                positive       145
                unlabelled    5457
Tethys          negative        21
                positive       481
                unlabelled    6507
dtype: int64

### Rename training data files

If `use_mantle_features == True`, rename the training data with mantle features to `training_data_global.csv`, enabling later notebooks (`01*` onwards) to easily read training data.

In [ ]:
if params["use_mantle_features"]:
    # Backup old data
    try:
        os.rename(data_dir / "training_data_global.csv", data_dir / "training_data_no_mantle.csv")
    except FileNotFoundError:
        pass
    # Save new data with mantle features
    training_data.to_csv(data_dir / "training_data_global.csv", index=False)